In [1]:
import sparknlp
sparknlp.version()


'5.1.4'

In [2]:
# RESTART with Docker-safe config
from sparknlp import start

# We lower heap memory to 2G to leave space for TensorFlow (C++) overhead
# We explicitly enable off-heap memory for Spark NLP
spark = start(params={
    "spark.driver.memory": "2g",               # REDUCED from 3g
    "spark.executor.memory": "2g",             # REDUCED from 3g
    "spark.executor.instances": "1",           # REDUCED to 1 (Single container cannot handle 2 executors + driver)
    "spark.executor.cores": "2",               
    "spark.sql.shuffle.partitions": "200",     # INCREASED: 30 is too low for 20GB data; creates massive partitions
    "spark.driver.maxResultSize": "1g",
    "spark.memory.offHeap.enabled": "true",    # IMPORTANT for NLP
    "spark.memory.offHeap.size": "2g",         # Reserve 2g for TensorFlow outside Java Heap
    "spark.sql.adaptive.enabled": "true",
    "spark.serializer": "org.apache.spark.serializer.KryoSerializer",
    "spark.kryoserializer.buffer.max": "2000M",
    "spark.network.timeout": "1200s",
    "spark.johnsnowlabs.nlp.pretrained.cache_folder": "/tmp/spark_nlp"
})

In [3]:
def output_exists(path):
    try:
        return spark.read.parquet(path).limit(1).count() > 0
    except:
        return False


In [4]:
from sparknlp.base import DocumentAssembler
from sparknlp.annotator import (
    Tokenizer,
    WordEmbeddingsModel,
    SentenceEmbeddings,
    SentimentDLModel
)
from pyspark.ml import Pipeline
from pyspark.sql.functions import col, when


In [5]:
# 1. Document
document = DocumentAssembler() \
    .setInputCol("text") \
    .setOutputCol("document")

# 2. Tokenizer
tokenizer = Tokenizer() \
    .setInputCols(["document"]) \
    .setOutputCol("token")

# 3. GloVe word embeddings
word_embeddings = WordEmbeddingsModel.pretrained(
    "glove_100d", "en"
).setInputCols(["document", "token"]) \
 .setOutputCol("word_embeddings")

# 4. Sentence embeddings (average pooling)
sentence_embeddings = SentenceEmbeddings() \
    .setInputCols(["document", "word_embeddings"]) \
    .setOutputCol("sentence_embeddings") \
    .setPoolingStrategy("AVERAGE")

# 5. SentimentDL (IMDB-trained)
sentiment = SentimentDLModel.pretrained(
    "sentimentdl_glove_imdb", "en"
).setInputCols(["sentence_embeddings"]) \
 .setOutputCol("sentiment")

pipeline = Pipeline(stages=[
    document,
    tokenizer,
    word_embeddings,
    sentence_embeddings,
    sentiment
])


glove_100d download started this may take some time.
Approximate size to download 145.3 MB
[OK!]
sentimentdl_glove_imdb download started this may take some time.
Approximate size to download 8.7 MB
[OK!]


In [6]:
remaining_states = [
    "Alabama","Alaska","Arizona","Arkansas","Colorado",
    "Connecticut","Delaware","District_of_Columbia", "Hawaii",
    "Idaho","Illinois","Indiana","Iowa","Kansas","Kentucky","Louisiana",
    "Maine","Maryland","Massachusetts","Michigan","Minnesota",
    "Mississippi","Missouri","Montana","Nebraska","Nevada",
    "New_Hampshire","New_Jersey","New_Mexico","North_Carolina",
    "North_Dakota","Ohio","Oklahoma","Oregon","Pennsylvania",
    "Rhode_Island","South_Carolina","South_Dakota",
    "Tennessee","Utah","Vermont","Virginia","Washington",
    "West_Virginia","Wisconsin","Wyoming","Texas","Florida","California","Georiga","New_York"
]

In [7]:
import gc
from pyspark.sql.functions import col
import time

def safe_exists(path):
    try:
        fs = spark._jvm.org.apache.hadoop.fs.FileSystem.get(spark._jsc.hadoopConfiguration())
        return fs.exists(spark._jvm.org.apache.hadoop.fs.Path(path))
    except:
        return False

# Refit model (small sample)
print("Refitting model...")
# Use a smaller sample to fit quickly
sample_df = spark.read.parquet("hdfs://namenode:8020/google_local/processed/state=Alabama").limit(100)
sentiment_model = pipeline.fit(sample_df)

for state in remaining_states:
    print(f"\n{'='*60}")
    print(f" STATE: {state}")
    print(f"{'='*60}")
    
    base_path = f"hdfs://namenode:8020/google_local/processed/state={state}"
    
    # 1. Fetch Years Optimized
    # We try to read only metadata if possible
    try:
        years = [row.year for row in spark.read.parquet(base_path).select("year").distinct().collect()]
        years.sort()
    except Exception as e:
        print(f"  Could not list years for {state}: {e}")
        continue
        
    print(f"  Found years: {years}")

    for y in years:
        output_path = f"hdfs://namenode:8020/google_local/Updated_Sentiment_Analysis/state={state}/year={y}"
        
        if output_exists(output_path):
             print(f"  Year {y} already exists, skipping")
             continue

        print(f"  Processing {y}...")
        
        try:
            # 2. Optimized Pipeline
            # We removed coalesce(8) to allow more/smaller partitions
            df_year = spark.read.parquet(base_path) \
                .filter(col("year") == y) \
                .filter(col("text").isNotNull()) \
                .dropDuplicates(["gmap_id", "text"])
            
            # Repartition to ensure partitions are small enough for NLP (~100MB each)
            # 200 partitions is safer for memory than 8
            df_year = df_year.repartition(50) 

            result = sentiment_model.transform(df_year) \
                .withColumn("sentiment_label", col("sentiment")[0]["result"]) \
                .select("gmap_id", "name", "rating", "sentiment_label", 
                       "avg_rating", "num_of_reviews", "category", "state", "year", "review_date")
            
            result.write.mode("overwrite").parquet(output_path)
            
            print(f"  {y} COMPLETED")
            
            # 3. CRITICAL: Free memory
            del df_year
            del result
            spark.catalog.clearCache()
            gc.collect() # Force Python Garbage Collection
            
        except Exception as e:
            print(f"  {y} FAILED: {str(e)[:100]}")
            # If it fails, try to restart spark context in real production, 
            # but here we just continue to try to save the next year
            continue

Refitting model...

 STATE: Alabama
  Found years: [2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021]
  Year 2007 already exists, skipping
  Year 2008 already exists, skipping
  Year 2009 already exists, skipping
  Year 2010 already exists, skipping
  Year 2011 already exists, skipping
  Year 2012 already exists, skipping
  Year 2013 already exists, skipping
  Year 2014 already exists, skipping
  Year 2015 already exists, skipping
  Year 2016 already exists, skipping
  Year 2017 already exists, skipping
  Year 2018 already exists, skipping
  Year 2019 already exists, skipping
  Year 2020 already exists, skipping
  Year 2021 already exists, skipping

 STATE: Alaska
  Found years: [2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021]
  Year 2007 already exists, skipping
  Year 2008 already exists, skipping
  Year 2009 already exists, skipping
  Year 2010 already exists, skipping
  Year 2011 already exists, skipp

In [7]:
df = spark.read.parquet(
    "hdfs://namenode:8020/google_local/Updated_Sentiment_Analysis/state=Arizona/year=2019"
)

In [8]:
df.select(
    "rating",
    "sentiment_label"
).show(20, truncate=False)


+------+---------------+
|rating|sentiment_label|
+------+---------------+
|5     |pos            |
|5     |pos            |
|4     |pos            |
|5     |pos            |
|3     |pos            |
|5     |pos            |
|1     |neg            |
|1     |pos            |
|1     |neg            |
|2     |neg            |
|4     |pos            |
|4     |pos            |
|5     |pos            |
|5     |pos            |
|5     |pos            |
|5     |pos            |
|1     |neg            |
|5     |pos            |
|5     |pos            |
|5     |pos            |
+------+---------------+
only showing top 20 rows

